# 03 — Preprocessing & Feature Engineering
**Objectif :** transformer le dataset EDA en matrice numérique prête pour XGBoost / scikit-learn.

**Étapes :**
1. Drop des features redondantes (`oltv` colinéaire avec `ocltv`)
2. Feature engineering sur les dates (YYYYMM → année, mois, durée résiduelle)
3. Encoding des catégorielles
   - Haute cardinalité (`property_state`, `msa`, `postal_code`) → **target encoding** (sur le train uniquement, pour éviter le leakage)
   - Basse cardinalité → **one-hot encoding**
4. Imputation des valeurs manquantes (médiane pour les numériques)
5. Split train/test stratifié (taux de défaut conservé dans les deux sets)

**Sortie :** `X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

DATA_PATH = os.path.join('..', 'data', 'freddie_mac_features.parquet')
df = pd.read_parquet(DATA_PATH)
print(f"Dataset chargé : {df.shape[0]:,} prêts × {df.shape[1]} colonnes")
print(f"Taux de défaut : {df['default'].mean():.2%}")
df.dtypes

Dataset chargé : 100,000 prêts × 24 colonnes
Taux de défaut : 5.57%


credit_score                 float64
first_payment_date            object
first_time_homebuyer_flag     object
maturity_date                 object
msa                           object
mip                            int64
units                          int64
occupancy_status              object
ocltv                        float64
dti                          float64
original_upb                   int64
oltv                         float64
original_interest_rate       float64
channel                       object
property_state                object
property_type                 object
postal_code                   object
loan_purpose                  object
original_loan_term             int64
number_of_borrowers           object
program_indicator             object
property_valuation_method     object
mi_cancellation_indicator     object
default                        int64
dtype: object

## 1. Drop des features redondantes

In [2]:
# oltv ↔ ocltv corrélés à 0.99 (EDA section 7) → on garde ocltv qui est plus complet
df = df.drop(columns=['oltv'])

# postal_code : cardinalité trop élevée (milliers de zones, beaucoup à <10 prêts).
# Le target encoding y mémorise le train sans généraliser (cf. SHAP : driver #1
# alors que l'AUC plafonne). Redondant avec msa ⊂ property_state qui sont plus stables.
df = df.drop(columns=['postal_code'])

print(f"Après drop : {df.shape[1]} colonnes")

Après drop : 22 colonnes


## 2. Feature engineering sur les dates

`first_payment_date` et `maturity_date` sont au format YYYYMM (entier).
On en extrait l'année et le mois, et on calcule `loan_term_years` = écart entre maturité et 1er paiement.

In [3]:
def yyyymm_to_year(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors='coerce') // 100

def yyyymm_to_month(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors='coerce') % 100

df['origination_year'] = yyyymm_to_year(df['first_payment_date'])
df['origination_month'] = yyyymm_to_month(df['first_payment_date'])
df['maturity_year'] = yyyymm_to_year(df['maturity_date'])
df['loan_term_years'] = df['maturity_year'] - df['origination_year']

# Les dates brutes ne servent plus
df = df.drop(columns=['first_payment_date', 'maturity_date', 'maturity_year'])

print(df[['origination_year', 'origination_month', 'loan_term_years']].describe().round(2))

       origination_year  origination_month  loan_term_years
count         100000.00          100000.00        100000.00
mean            2017.66               6.66            27.36
std                0.62               3.45             5.52
min             2017.00               1.00             8.00
25%             2017.00               4.00            29.00
50%             2018.00               7.00            30.00
75%             2018.00              10.00            30.00
max             2021.00              12.00            30.00


## 2.5 Feature engineering métier

On crée des variables dérivées qui encodent du **savoir métier crédit** :

- **Flags de risque** : seuils réglementaires/empiriques (FICO subprime <660, LTV >95, DTI >43)
- **Risk score additif** : nombre de critères de risque cumulés (0 à 3)
- **Interactions** : FICO × DTI, FICO × LTV — captent les profils cumulant plusieurs risques
- **Mensualité approximée** : formule du prêt à mensualité constante → proxy de l'effort financier
- **Spread de taux** : écart entre le taux du prêt et le taux moyen du millésime

Ces features aident à la fois :
- la **LogReg** (transforme les non-linéarités en linéarités après bucketing)
- **XGBoost** (réduit la profondeur nécessaire pour capturer les interactions)

In [4]:
# Conversion préalable des variables qu'on va utiliser (avant la section 3)
for col in ['credit_score', 'dti', 'ocltv', 'original_upb',
            'original_interest_rate', 'original_loan_term']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 1. Flags de risque (seuils métier classiques en US mortgage)
df['is_subprime']    = (df['credit_score'] < 660).astype(int)
df['is_high_ltv']    = (df['ocltv'] > 95).astype(int)
df['is_high_dti']    = (df['dti'] > 43).astype(int)
df['risk_count']     = df['is_subprime'] + df['is_high_ltv'] + df['is_high_dti']

# 2. Interactions — formulées pour que "plus haut = plus risqué"
# (850 - FICO) donne un "déficit de score" qu'on multiplie par DTI/LTV
df['fico_deficit']        = (850 - df['credit_score']).clip(lower=0)
df['fico_dti_interaction'] = df['fico_deficit'] * df['dti'] / 100
df['fico_ltv_interaction'] = df['fico_deficit'] * df['ocltv'] / 100
df['dti_ltv_interaction']  = df['dti'] * df['ocltv'] / 100

# 3. Mensualité approximée — formule du prêt à amortissement constant
# M = P × r(1+r)^n / ((1+r)^n - 1)  avec r mensuel et n en mois
r_monthly = df['original_interest_rate'] / 100 / 12
n_months  = df['original_loan_term']
P         = df['original_upb']
# Cas r=0 traité séparément (rare)
factor = ((1 + r_monthly) ** n_months)
df['monthly_payment'] = np.where(
    r_monthly > 0,
    P * r_monthly * factor / (factor - 1),
    P / n_months
)

# 4. Spread de taux : écart par rapport à la moyenne du millésime
mean_rate_by_year = df.groupby('origination_year')['original_interest_rate'].transform('mean')
df['rate_spread'] = df['original_interest_rate'] - mean_rate_by_year

# 5. Loan-to-payment ratio : indicateur d'effort financier rapporté au prêt
df['payment_to_upb_ratio'] = df['monthly_payment'] / df['original_upb']

# Récap
new_features = ['is_subprime', 'is_high_ltv', 'is_high_dti', 'risk_count',
                'fico_deficit', 'fico_dti_interaction', 'fico_ltv_interaction',
                'dti_ltv_interaction', 'monthly_payment', 'rate_spread',
                'payment_to_upb_ratio']

print(f"✓ {len(new_features)} nouvelles features créées")
print(f"\nDistribution du risk_count :")
print(df['risk_count'].value_counts().sort_index())
print(f"\nTaux de défaut par risk_count :")
print(df.groupby('risk_count')['default'].agg(['mean', 'count']).round(4))
print(f"\nStats des nouvelles features numériques :")
print(df[new_features].describe().round(2).T)

✓ 11 nouvelles features créées

Distribution du risk_count :
risk_count
0    70113
1    27771
2     2106
3       10
Name: count, dtype: int64

Taux de défaut par risk_count :
              mean  count
risk_count               
0           0.0403  70113
1           0.0869  27771
2           0.1586   2106
3           0.2000     10

Stats des nouvelles features numériques :
                         count     mean     std    min     25%      50%  \
is_subprime           100000.0     0.05    0.21   0.00    0.00     0.00   
is_high_ltv           100000.0     0.04    0.21   0.00    0.00     0.00   
is_high_dti           100000.0     0.23    0.42   0.00    0.00     0.00   
risk_count            100000.0     0.32    0.51   0.00    0.00     0.00   
fico_deficit           99978.0   104.53   46.65  16.00   66.00    97.00   
fico_dti_interaction   97659.0    37.41   20.18   0.41   21.16    33.88   
fico_ltv_interaction   99977.0    79.70   40.14   2.70   47.53    74.10   
dti_ltv_interaction    976

## 3. Conversion des types et imputation des manquantes

In [5]:
# Les colonnes type 'object' qui devraient être numériques
NUMERIC_AS_STR = ['mip', 'units', 'number_of_borrowers', 'original_loan_term',
                  'program_indicator', 'property_valuation_method', 'mi_cancellation_indicator']
for col in NUMERIC_AS_STR:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Imputation des numériques par la médiane (sauf default)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'default']

for col in numeric_cols:
    median = df[col].median()
    df[col] = df[col].fillna(median)

print(f"Numériques imputées par médiane : {len(numeric_cols)} colonnes")
print(f"Manquants restants par colonne :")
missing = df.isnull().sum()
print(missing[missing > 0])

Numériques imputées par médiane : 26 colonnes
Manquants restants par colonne :
msa    10572
dtype: int64


## 4. Split train/test stratifié

**Important :** on split AVANT d'encoder les catégorielles haute cardinalité, sinon le target encoding fuit l'information du test set vers le train.

In [6]:
X = df.drop(columns=['default'])
y = df['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train : {X_train.shape[0]:,} prêts | défaut = {y_train.mean():.2%}")
print(f"Test  : {X_test.shape[0]:,} prêts | défaut = {y_test.mean():.2%}")

Train : 80,000 prêts | défaut = 5.57%
Test  : 20,000 prêts | défaut = 5.57%


## 5. Target encoding (haute cardinalité)

Pour `property_state` et `msa` : on remplace chaque modalité par le **taux de défaut moyen observé sur le train**.

`postal_code` a été retiré (section 1) : trop granulaire, il mémorisait le train sans généraliser. `msa` et `property_state` sont plus grossiers, donc chaque modalité a assez d'observations pour une estimation fiable.

Astuce anti-overfit : on lisse avec un *smoothing* qui ramène vers la moyenne globale les modalités rares (peu d'observations → estimation peu fiable).

In [7]:
def target_encode(train_col: pd.Series, test_col: pd.Series, target: pd.Series,
                  smoothing: float = 10.0):
    """
    Encode une variable catégorielle par le taux de défaut moyen, lissé.
    Formule : (count * mean_modalité + smoothing * mean_global) / (count + smoothing)
    Retourne aussi le mapping {modalité: valeur} et la moyenne globale (pour l'app Streamlit).
    """
    global_mean = target.mean()
    agg = target.groupby(train_col).agg(['count', 'mean'])
    smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
    mapping = smooth.to_dict()

    train_encoded = train_col.map(mapping).fillna(global_mean)
    test_encoded = test_col.map(mapping).fillna(global_mean)  # modalités inconnues → moyenne globale
    return train_encoded, test_encoded, mapping, global_mean

# postal_code retiré : trop granulaire. On garde msa + property_state (plus stables).
HIGH_CARDINALITY = ['property_state', 'msa']

# On conserve les mappings pour les réutiliser dans l'app Streamlit
target_encoding_maps = {}
target_global_mean = y_train.mean()

for col in HIGH_CARDINALITY:
    if col in X_train.columns:
        X_train[col], X_test[col], mapping, gmean = target_encode(
            X_train[col].astype(str), X_test[col].astype(str), y_train
        )
        target_encoding_maps[col] = mapping
        print(f"  {col} → target-encoded ({len(mapping)} modalités)")

print(f"\nDtypes après target encoding :")
print(X_train[HIGH_CARDINALITY].dtypes)

  property_state → target-encoded (54 modalités)
  msa → target-encoded (441 modalités)

Dtypes après target encoding :
property_state    float64
msa               float64
dtype: object


## 6. One-hot encoding (basse cardinalité)

In [8]:
LOW_CARDINALITY = ['channel', 'occupancy_status', 'property_type',
                   'loan_purpose', 'first_time_homebuyer_flag']
LOW_CARDINALITY = [c for c in LOW_CARDINALITY if c in X_train.columns]

# concat → encode → split, pour garantir les mêmes colonnes dans train et test
X_train['__split'] = 'train'
X_test['__split'] = 'test'
combined = pd.concat([X_train, X_test], axis=0)
combined = pd.get_dummies(combined, columns=LOW_CARDINALITY, drop_first=True, dtype=int)

X_train = combined[combined['__split'] == 'train'].drop(columns='__split')
X_test = combined[combined['__split'] == 'test'].drop(columns='__split')

print(f"X_train : {X_train.shape[1]} colonnes après one-hot")
print(f"X_test  : {X_test.shape[1]} colonnes après one-hot")
print(f"\nNouvelles colonnes one-hot (exemple) :")
ohe_cols = [c for c in X_train.columns if any(c.startswith(p + '_') for p in LOW_CARDINALITY)]
print(ohe_cols)

X_train : 39 colonnes après one-hot
X_test  : 39 colonnes après one-hot

Nouvelles colonnes one-hot (exemple) :
['channel_C', 'channel_R', 'occupancy_status_P', 'occupancy_status_S', 'property_type_CP', 'property_type_MH', 'property_type_PU', 'property_type_SF', 'loan_purpose_N', 'loan_purpose_P', 'first_time_homebuyer_flag_Y']


## 7. Vérification finale : tout doit être numérique

In [9]:
non_numeric = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print(f"⚠ Colonnes non numériques restantes : {non_numeric}")
    for col in non_numeric:
        print(f"  {col} : {X_train[col].unique()[:5]}")
else:
    print("✓ Toutes les colonnes sont numériques")

print(f"\nShape final :")
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train : {y_train.shape} (défaut = {y_train.mean():.2%})")
print(f"  y_test  : {y_test.shape} (défaut = {y_test.mean():.2%})")

✓ Toutes les colonnes sont numériques

Shape final :
  X_train : (80000, 39)
  X_test  : (20000, 39)
  y_train : (80000,) (défaut = 5.57%)
  y_test  : (20000,) (défaut = 5.57%)


## 8. Sauvegarde

In [10]:
out_dir = os.path.join('..', 'data')
X_train.to_parquet(os.path.join(out_dir, 'X_train.parquet'), index=False)
X_test.to_parquet(os.path.join(out_dir, 'X_test.parquet'), index=False)
y_train.to_frame('default').to_parquet(os.path.join(out_dir, 'y_train.parquet'), index=False)
y_test.to_frame('default').to_parquet(os.path.join(out_dir, 'y_test.parquet'), index=False)

print("Sauvegardé dans data/ :")
for f in ['X_train', 'X_test', 'y_train', 'y_test']:
    path = os.path.join(out_dir, f + '.parquet')
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f}.parquet ({size_mb:.1f} MB)")

Sauvegardé dans data/ :
  X_train.parquet (2.0 MB)
  X_test.parquet (0.6 MB)
  y_train.parquet (0.0 MB)
  y_test.parquet (0.0 MB)


## 9. Sauvegarde du préprocesseur (pour l'app Streamlit)

L'app de scoring reçoit un profil brut (FICO, DTI, état, etc.) et doit reconstruire le vecteur de 39 features attendu par le modèle. On sauvegarde donc tout ce qu'il faut pour rejouer le pipeline à l'inférence :
- les mappings de target encoding (`property_state`, `msa`)
- les médianes d'imputation des variables numériques
- l'ordre exact des colonnes du modèle

In [11]:
import pickle

MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# Médianes des colonnes numériques du modèle (valeurs par défaut à l'inférence)
feature_medians = X_train.median(numeric_only=True).to_dict()

preprocessor = {
    'feature_columns': list(X_train.columns),          # ordre exact attendu par le modèle
    'target_encoding_maps': target_encoding_maps,       # {col: {modalité: valeur}}
    'target_global_mean': float(target_global_mean),    # fallback modalités inconnues
    'feature_medians': feature_medians,                 # défauts d'imputation
}

with open(os.path.join(MODELS_DIR, 'preprocessor.pkl'), 'wb') as f:
    pickle.dump(preprocessor, f)

print('Sauvegardé : models/preprocessor.pkl')
print(f"  {len(preprocessor['feature_columns'])} colonnes")
print(f"  target encoding : {list(target_encoding_maps.keys())}")
print(f"  global mean (fallback) : {target_global_mean:.4f}")

Sauvegardé : models/preprocessor.pkl
  39 colonnes
  target encoding : ['property_state', 'msa']
  global mean (fallback) : 0.0557
